<a href="https://colab.research.google.com/github/amriT2044524/InSAR_Himalaya/blob/main/radar_and_shadow_mask_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/ASFHyP3/hyp3-sdk/blob/main/docs/sdk_example.ipynb

In [ ]:
!pip install hyp3_sdk
# !pip install -U rasterio rioxarray
!pip install asf_search

In [ ]:
# initial setup
import asf_search as asf
import hyp3_sdk as sdk
import glob
import os
import zipfile
import xarray as xr
import rioxarray as rxr
import pandas as pd
import numpy as np

from IPython.display import display
import ipywidgets as widgets

In [ ]:
# prompt for your EDL username and password
hyp3 = sdk.HyP3(prompt='password')

In [ ]:
granules = [
    'S1A_IW_SLC__1SDV_20260115T123914_20260115T123941_062778_07DF9E_5528',
    'S1A_IW_SLC__1SDV_20260103T123915_20260103T123942_062603_07D8DE_9271',
]

rtc_jobs = sdk.Batch()
for g in granules:
    rtc_jobs += hyp3.submit_rtc_job(g, name='rtc-example')
print(rtc_jobs)

In [ ]:
rtc_jobs = hyp3.watch(rtc_jobs)

In [ ]:
file_list = rtc_jobs.download_files()

In [ ]:
zip_files = glob.glob("*.zip")
zip_files


In [ ]:
names_no_zip = [os.path.splitext(f)[0] for f in zip_files]
names_no_zip

In [ ]:
i=0

In [ ]:
zip_path = f"{names_no_zip[i]}.zip"
out_dir = f"{names_no_zip[i]}"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(out_dir)

# list files
os.listdir(out_dir)


In [ ]:
tif_shadow=f"/content/{out_dir}/{out_dir}/{names_no_zip[i]}_ls_map.tif"
tif_shadow

In [ ]:
da = rxr.open_rasterio(tif_shadow)
da


In [ ]:
# da.plot()

In [ ]:
import rioxarray as rxr
from pyproj import Transformer
from rasterio.enums import Resampling

# WGS84 bbox (lon/lat), define AOI
lon_min, lon_max, lat_min, lat_max = 81.3998, 82.16178, 29.7501, 30.51603

# 1) Ensure da has CRS (must not be None)
print("da CRS:", da.rio.crs)
# If None, you MUST set it correctly first, e.g.:
# da = da.rio.write_crs("EPSG:32644")  # <-- example, replace with correct CRS

src_crs = da.rio.crs
dst_crs = "EPSG:4326"

# 2) Transform bbox corners from EPSG:4326 -> da CRS
tfm = Transformer.from_crs(dst_crs, src_crs, always_xy=True)

x1, y1 = tfm.transform(lon_min, lat_min)
x2, y2 = tfm.transform(lon_max, lat_max)

minx, maxx = sorted([x1, x2])
miny, maxy = sorted([y1, y2])

In [ ]:
# 3) Crop in native projection (fast)
da_crop_native = da.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)


In [ ]:
da_crop_native.plot()

In [ ]:
# 4) Reproject the cropped raster back to WGS84
da_crop_wgs84 = da_crop_native.rio.reproject(dst_crs, resampling=Resampling.nearest)
da_crop_wgs84

In [ ]:
da_crop_wgs84.plot()

In [ ]:
df = da_crop_wgs84.to_dataframe(name="value").reset_index()
df.head()


In [ ]:
df["value"].dropna().unique().tolist()

value=c(1,5,17,21)

desc=c('No Layover/Shadow','Layover','Shadow','Layover+Shadow'))


In [ ]:
shadow_layover_mask = (da_crop_wgs84 == 1).astype("uint8")

In [ ]:
np.unique(shadow_layover_mask.values)

In [ ]:
from google.colab import files

out_file = f"shadow_layover_mask_{names_no_zip[i]}.tif"
shadow_layover_mask.rio.to_raster(out_file)
files.download(out_file)


In [ ]:
shadow_layover_mask.plot()

In [ ]:
import numpy as np
import folium

# --- ensure 2D mask array ---
arr = np.asarray(shadow_layover_mask.values)
if arr.ndim == 3:
    arr = arr[0]  # (y, x)
arr = (arr == 1).astype(np.uint8)  # force binary 0/1

# --- create RGBA image ---
rgba = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)
rgba[..., 0] = 255                 # Red channel
rgba[..., 3] = arr * 180           # Alpha: 0 for 0s, ~70% for 1s (0-255)

# bounds
bounds = [
    [float(shadow_layover_mask.y.min()), float(shadow_layover_mask.x.min())],
    [float(shadow_layover_mask.y.max()), float(shadow_layover_mask.x.max())]
]

# map
lat0 = float(shadow_layover_mask.y.mean())
lon0 = float(shadow_layover_mask.x.mean())
# m = folium.Map(location=[lat0, lon0], zoom_start=9, tiles="OpenStreetMap")
m = folium.Map(location=[lat0, lon0], zoom_start=9, tiles="Esri.WorldImagery")


folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=bounds,
    opacity=1.0,   # keep 1.0 because alpha is handled in RGBA
).add_to(m)

m


## 6. Layover-shadow mask <read read me file >

The layover/shadow mask indicates which pixels in the RTC image have been affected by layover and shadow. This layer is always included in the product package, and is tagged with _ls_map.tif

The pixel values are generated by adding the following values together to indicate which layover and shadow effects are impacting each pixel:
0  Pixel not tested for layover or shadow

1  Pixel tested for layover or shadow

2  Pixel has a look angle less than the slope angle

4  Pixel is in an area affected by layover

8  Pixel has a look angle less than the opposite of the slope angle

16 Pixel is in an area affected by shadow

There are 17 possible different pixel values, indicating the layover, shadow, and slope conditions present added together for any given pixel._

**The values in each cell can range from 0 to 31:**
0  Not tested for layover or shadow
1  Not affected by either layover or shadow
3  Look angle < slope angle
5  Affected by layover
7  Affected by layover; look angle < slope angle
9  Look angle < opposite slope angle
11 Look angle < slope and opposite slope angle
13 Affected by layover; look angle < opposite slope angle
15 Affected by layover; look angle < slope and opposite slope angle
17 Affected by shadow
19 Affected by shadow; look angle < slope angle
21 Affected by layover and shadow
23 Affected by layover and shadow; look angle < slope angle
25 Affected by shadow; look angle < opposite slope angle
27 Affected by shadow; look angle < slope and opposite slope angle
29 Affected by shadow and layover; look angle < opposite slope angle
31 Affected by shadow and layover; look angle < slope and opposite slope angle

-------------

https://hyp3-docs.asf.alaska.edu/guides/rtc_product_guide/#image-files